# SetUp

In [85]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("../../")
    
print(f"Directorio de trabajo actual: {os.getcwd()}")

import jax
import jax.numpy as jnp
import numpy as np

DATA_PATH = "crystals/"
CIF_PATH = DATA_PATH + "cif/"
METADATA_PATH = DATA_PATH + "_metadata.json"
EMBEDDINGS_PATH = DATA_PATH + "_embeddings.json"
FINAL_DATA_PATH = DATA_PATH + "_data.json"
DATASET_PATH = DATA_PATH + "dataset_fase1a.h5" 

Directorio de trabajo actual: /home/alanh/Dev/owns/thesis


In [86]:
# load metadata
import json 
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)
materiales_dict = {m['material_id']: m for m in metadata}
del metadata

# Loss Functions

## MSE Loss

In [87]:
def crystal_loss_fn1(preds, y_target, lattice_params=6, max_atoms=4):
    # 1. Separación de datos
    pred_lattice   = preds[:, :lattice_params]
    target_lattice = y_target[:, :lattice_params]

    pred_atoms   = preds[:, lattice_params:].reshape(-1, max_atoms, 4)
    target_atoms = y_target[:, lattice_params:].reshape(-1, max_atoms, 4)

    mask = (target_atoms[:, :, 0] > 0).astype(jnp.float32)

    # --- LOSSES PUNTUALES (Individuo por Individuo) ---
    lat_loss = jnp.mean((pred_lattice - target_lattice) ** 2)
    z_loss   = jnp.mean((pred_atoms[:, :, 0] - target_atoms[:, :, 0]) ** 2)
    
    diff = jnp.abs(pred_atoms[:, :, 1:] - target_atoms[:, :, 1:])
    pbc_diff = jnp.minimum(diff, 1.0 - diff) 
    pos_loss = jnp.mean(mask[:, :, None] * (pbc_diff ** 2))

    oob = jnp.mean(mask[:, :, None] * (
        jnp.maximum(0.0, pred_atoms[:, :, 1:] - 1.0) ** 2 +
        jnp.maximum(0.0, -pred_atoms[:, :, 1:]) ** 2
    ))

    # --- LOSSES DE DISTRIBUCIÓN (Batch completo) ---
    
    # A. Varianza de las predicciones vs Varianza de los targets
    # Calculamos qué tanto varían los parámetros en este batch
    pred_var   = jnp.var(pred_lattice, axis=0)   # Varianza predicha (6 params)
    target_var = jnp.var(target_lattice, axis=0) # Varianza real (6 params)
    
    # Loss de Emparejamiento: Queremos que la diferencia entre varianzas sea cero
    # Usamos un epsilon para estabilidad
    var_matching_loss = jnp.mean((pred_var - target_var) ** 2)

    # B. Penalización de colapso (Repulsión pura)
    # Sigue siendo útil como "red de seguridad" para evitar que la var_matching 
    # se quede atrapada en 0 si el batch es muy pequeño.
    collapse_penalty = jnp.mean(jnp.exp(-1000.0 * (pred_var + 1e-6)))

    # --- TOTAL REBALANCEADO ---
    # Pesos sugeridos para obligar a NEAT a salir del colapso:
    return (
        (20.0 * lat_loss) +          # Precisión de celda
        (5.0  * var_matching_loss) + # 🚀 Alineación de diversidad
        (5.0  * z_loss) +            # Elementos químicos
        (20.0 * pos_loss) +          # Estructura atómica
        (10.0 * oob) +               # Guardarraíl de coordenadas
        (30.0 * collapse_penalty)    # Repulsión de emergencia
    )

## Graph Loss

In [88]:
def crystal_loss_fn2(preds, y_target, lattice_params=6, max_atoms=4):
    """LA NUEVA: Mini-ACSF (Invariante) + Composición + Repulsión"""
    pred_lat = preds[:, :lattice_params]
    target_lat = y_target[:, :lattice_params]
    
    pred_atoms = preds[:, lattice_params:].reshape(-1, max_atoms, 4)
    target_atoms = y_target[:, lattice_params:].reshape(-1, max_atoms, 4)
    
    # Extraemos los números atómicos (Z)
    pred_z = pred_atoms[:, :, 0]
    target_z = target_atoms[:, :, 0]
    
    mask = (target_z > 0).astype(jnp.float32)
    mask_2d = mask[:, :, None] * mask[:, None, :]
    
    # ---------------------------------------------------------
    # 1. LATTICE (Tamaño de la caja)
    # ---------------------------------------------------------
    lat_loss = jnp.mean((pred_lat - target_lat)**2)

    # ---------------------------------------------------------
    # 2. ACSF (Invarianza Geométrica de la Estructura)
    # ---------------------------------------------------------
    def get_sorted_fingerprints(pos, m2d, eta=5.0):
        diff = pos[:, :, None, :] - pos[:, None, :, :]
        diff = diff - jnp.round(diff)
        dist_sq = jnp.sum(diff**2, axis=-1)
        gaussians = jnp.exp(-eta * dist_sq) * m2d
        fingerprints = jnp.sum(gaussians, axis=-1)
        return jnp.sort(fingerprints, axis=-1)

    fp_pred = get_sorted_fingerprints(pred_atoms[:, :, 1:], mask_2d)
    fp_target = get_sorted_fingerprints(target_atoms[:, :, 1:], mask_2d)
    acsf_loss = jnp.mean((fp_pred - fp_target)**2)

    # ---------------------------------------------------------
    # 3. Z-LOSS INVARIANTE (El castigo para los "Hermanos")
    # ---------------------------------------------------------
    # Ordenamos los elementos químicos presentes en la celda.
    # Si predice [Cl, Na] y el target es [Na, Cl], al ordenarlos ambos
    # quedan como [Cl, Na], dando un error de 0.0. Invarianza perfecta.
    # Usamos la máscara en la predicción para ignorar el "ruido" en posiciones vacías.
    sorted_pred_z = jnp.sort(pred_z * mask, axis=1)
    sorted_target_z = jnp.sort(target_z, axis=1)
    z_loss = jnp.mean((sorted_pred_z - sorted_target_z)**2)

    # ---------------------------------------------------------
    # 4. VARIANCE & REPULSION (Anti-Colapso)
    # ---------------------------------------------------------
    fp_var_pred = jnp.var(fp_pred, axis=0)
    fp_var_target = jnp.var(fp_target, axis=0)
    var_loss = jnp.mean((fp_var_pred - fp_var_target)**2)
    repulsion_loss = jnp.mean(jnp.exp(-1000.0 * (fp_var_pred + 1e-6)))

    # ---------------------------------------------------------
    # PESOS 50/50 (Estructura vs Reglas Físicas)
    # ---------------------------------------------------------
    # Total = 100 puntos de Loss
    # 50% = ACSF (Forma)
    # 20% = Z_Loss (Química/Ingredientes estrictos)
    # 10% = Lat_Loss (Tamaño de la caja)
    # 20% = Varianza/Repulsión (Exploración)
    
    return (10.0 * lat_loss) + (50.0 * acsf_loss) + (20.0 * z_loss) + (10.0 * var_loss) + (10.0 * repulsion_loss)

# Test

In [103]:
import random

import h5py
import jax
import jax.numpy as jnp
import numpy as np

# ==============================================================================
# 1. CARGA DE DATOS REALES (Directo del H5)
# ==============================================================================
with h5py.File(DATASET_PATH, 'r') as hf:
    # Tomamos 2 cristales arbitrarios (ej. el índice 0 y el índice 10)
    # Tienen forma de vector plano (ej. 22 elementos: 6 lattice + 16 atoms)
    n1 = random.randint(0, 84)  # Índice aleatorio para el primer cristal
    n2 = random.randint(0, 84)  # Índice aleatorio para el segundo cristal
    # n1 = 0
    # n2 = 10
    c1_numpy = hf['targets'][n1]
    c2_numpy = hf['targets'][n2]
    c1_id = hf['material_ids'][n1].decode('utf-8')
    c2_id = hf['material_ids'][n2].decode('utf-8')

C1 = jnp.array(c1_numpy)
C2 = jnp.array(c2_numpy)
print(f"Cristal 1 ID: {c1_id} {materiales_dict.get(c1_id)['formula']} | Cristal 2 ID: {c2_id} {materiales_dict.get(c2_id)['formula']}")

# Creamos el target batch base. 
# Usamos C2 como el segundo elemento para que la Varianza no sea 0.0 y no explote la repulsión.
target_batch = jnp.stack([C1, C2])

def make_batch(modified_c1):
    """Empaqueta el C1 modificado con el C2 intacto para someterlo a la loss"""
    return jnp.stack([modified_c1, C2])

# ==============================================================================
# 2. TRANSFORMACIONES MATEMÁTICAS SOBRE EL VECTOR REAL
# ==============================================================================
# Índices XYZ en tu vector de 22:
# Átomo 1 (7,8,9), Átomo 2 (11,12,13), Átomo 3 (15,16,17), Átomo 4 (19,20,21)
xyz_idx = jnp.array([7, 8, 9, 11, 12, 13, 15, 16, 17, 19, 20, 21])

def rotate_z(c):
    """Rotación de 90° en Z: x -> -y, y -> x"""
    c_rot = c.copy()
    for base in [7, 11, 15, 19]:
        c_rot = c_rot.at[base].set(jnp.mod(-c[base+1], 1.0)) # x = -y
        c_rot = c_rot.at[base+1].set(c[base])               # y = x original
    return c_rot

def translate(c, shift=0.2):
    """Traslación con PBC"""
    c_trans = c.copy()
    c_trans = c_trans.at[xyz_idx].set(jnp.mod(c[xyz_idx] + shift, 1.0))
    return c_trans

def permute(c):
    """Intercambia completamente el Átomo 1 (índices 6:10) y el Átomo 2 (índices 10:14)"""
    c_perm = c.copy()
    c_perm = c_perm.at[6:10].set(c[10:14])
    c_perm = c_perm.at[10:14].set(c[6:10])
    return c_perm

def add_noise(c, noise_level=0.01):
    """Ruido Gaussiano a las coordenadas espaciales"""
    key = jax.random.PRNGKey(42)
    noise = jax.random.normal(key, xyz_idx.shape) * noise_level
    c_noisy = c.copy()
    c_noisy = c_noisy.at[xyz_idx].set(jnp.mod(c[xyz_idx] + noise, 1.0))
    return c_noisy

# ==============================================================================
# 3. EL RING DE COMBATE
# ==============================================================================

def run_duel(name, pred_batch, target_batch):
    # Evaluamos ambas funciones pasando el batch simulado
    loss1 = crystal_loss_fn1(pred_batch, target_batch)
    loss2 = crystal_loss_fn2(pred_batch, target_batch)
    print(f"{name:.<26} | MSE Antigua: {loss1:8.4f} | ACSF Nueva: {loss2:8.4f}")

print("🥊 AUDITORÍA DE INVARIANZA (DATOS REALES) 🥊\n")

run_duel("1. Idéntico", target_batch, target_batch)
run_duel("2. Rotado 90°", make_batch(rotate_z(C1)), target_batch)
run_duel("3. Trasladado (+0.2)", make_batch(translate(C1)), target_batch)
run_duel("4. Permutado (Swap 1x2)", make_batch(permute(C1)), target_batch)
run_duel("5. Ruido Pequeño (0.01)", make_batch(add_noise(C1, 0.01)), target_batch)
run_duel("6. Ruido Fuerte (0.10)", make_batch(add_noise(C1, 0.10)), target_batch)
run_duel("7. Cristal Distinto (C2)", make_batch(C2), target_batch)

Cristal 1 ID: mp-830 GaN | Cristal 2 ID: mp-22922 AgCl
🥊 AUDITORÍA DE INVARIANZA (DATOS REALES) 🥊

1. Idéntico............... | MSE Antigua:  15.0533 | ACSF Nueva:   4.9950
2. Rotado 90°............. | MSE Antigua:  15.2617 | ACSF Nueva:   4.9950
3. Trasladado (+0.2)...... | MSE Antigua:  15.2533 | ACSF Nueva:   4.9950
4. Permutado (Swap 1x2)... | MSE Antigua:  15.4175 | ACSF Nueva:   4.9950
5. Ruido Pequeño (0.01)... | MSE Antigua:  15.0534 | ACSF Nueva:   4.9953
6. Ruido Fuerte (0.10).... | MSE Antigua:  15.0566 | ACSF Nueva:   5.0287
7. Cristal Distinto (C2).. | MSE Antigua:  30.2501 | ACSF Nueva:  11.8072
